<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_43_crewai_autogen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🎭 **Module 43 — crewAI & AutoGen (multi-agent orchestration patterns)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 43 — crewAI & AutoGen

> A single LLM agent (M32 LangChain, M33 LangGraph) is one actor with tools. **Multi-agent** systems compose multiple specialised agents — a Researcher, a Writer, a Critic — that **talk to each other** to solve a task. The two leading frameworks: **crewAI** (role-based, declarative, easy to read) and **AutoGen** from Microsoft (conversation-based, programmable, built for tool-using agents).
>
> This module shows **the same task in both frameworks** so you can pick.

### What you'll cover
1. When multi-agent helps — and when it just adds tokens
2. The two mental models: **roles + tasks** (crewAI) vs **conversation graph** (AutoGen)
3. Setup — point both frameworks at local Ollama (free) or OpenAI
4. **crewAI** — Agent / Task / Crew with sequential + hierarchical processes
5. **AutoGen** — `AssistantAgent` + `UserProxyAgent` + GroupChat
6. Tool use — agents calling Python functions
7. Patterns: **Researcher → Writer → Critic** (sequential)
8. Patterns: **Manager + Workers** (hierarchical)
9. Common failure modes (loops, hallucinated handoffs, runaway cost)
10. When to use crewAI vs AutoGen vs LangGraph (M33)

> 🟡 We'll demo with **Ollama** so the notebook runs free. Same code with `OPENAI_API_KEY` set runs against GPT-4o.


## 1 · When multi-agent actually helps

Multi-agent is **not free** — every "agent talks to agent" turn is another LLM call. Use it when:

| ✅ Good fit | ❌ Bad fit |
|---|---|
| Tasks have **clear sub-roles** (research → write → review) | Single-shot Q&A; one LLM call is enough |
| You want **explicit critique / verification** of an output | Anywhere a chain-of-thought prompt would do |
| Workflow needs **specialised tools per agent** | Latency or cost is a hard constraint |
| The output benefits from **back-and-forth** | The task is deterministic — use code |

**Rule of thumb.** If you can't draw the role chart on the back of a napkin, you don't have a multi-agent problem yet — you have a prompting problem.


## 2 · Two mental models

| | **crewAI** | **AutoGen** |
|---|---|---|
| Mental model | **Crew of role-based experts** working a list of tasks | **Conversation** between agents that route messages |
| Style | Declarative (configure agents + tasks, run) | Programmatic (you wire conversations) |
| Strengths | Clear, readable, easy onboarding | Tool use, code execution, group chats with managers |
| Weaknesses | Less control over per-message routing | More code, more concepts |
| Killer feature | `Process.hierarchical` — one Manager LLM dispatches | `GroupChatManager` + Docker-sandboxed code execution |

In **LangGraph (M33)** you'd build the same thing as an explicit state machine. crewAI / AutoGen hide that machinery — at the cost of less control. Pick LangGraph when you need branching, retries, and time-travel; crewAI / AutoGen when you want fast iteration on the **roles** themselves.


## 3 · Setup — point at Ollama (free) or OpenAI

We assume Ollama is already running on `http://localhost:11434` (M38). If not, restart this section after running the M38 setup cells.

In [ ]:
!pip -q install crewai==0.86.0 "crewai-tools" pyautogen==0.4.0 ollama

In [ ]:
import os
# point both frameworks at Ollama via OpenAI-compat
os.environ["OPENAI_API_KEY"]   = "ollama"            # any non-empty string
os.environ["OPENAI_API_BASE"]  = "http://localhost:11434/v1"
os.environ["OPENAI_MODEL_NAME"]= "qwen2.5:0.5b-instruct"   # tiny, free, fast on CPU
print("ENV ready")

## 4 · crewAI — Researcher → Writer → Critic

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.llm import LLM

# Tell crewAI to talk to local Ollama through the OpenAI-compat shim
local_llm = LLM(model="openai/qwen2.5:0.5b-instruct",
                base_url="http://localhost:11434/v1",
                api_key="ollama")

researcher = Agent(
    role="Senior Researcher",
    goal="Find three concrete facts about the topic.",
    backstory="You are meticulous and cite sources.",
    llm=local_llm,
    verbose=True,
)
writer = Agent(
    role="Tech Writer",
    goal="Turn the researcher's bullets into a 5-sentence summary.",
    backstory="You write tight, no fluff.",
    llm=local_llm,
    verbose=True,
)
critic = Agent(
    role="Editor",
    goal="Find at least one weakness in the draft and suggest a fix.",
    backstory="You are direct and constructive.",
    llm=local_llm,
    verbose=True,
)

In [ ]:
topic = "The Transformer attention mechanism"

t_research = Task(description=f"Research the topic: {topic}. Output 3 bullet points.",
                  expected_output="3 bullet points with sources",
                  agent=researcher)
t_write    = Task(description="Write a 5-sentence summary using the bullets.",
                  expected_output="5-sentence paragraph",
                  agent=writer,
                  context=[t_research])
t_critique = Task(description="Critique the draft. Suggest one specific improvement.",
                  expected_output="Critique + suggestion",
                  agent=critic,
                  context=[t_write])

crew = Crew(agents=[researcher, writer, critic],
            tasks=[t_research, t_write, t_critique],
            process=Process.sequential,
            verbose=False)

# result = crew.kickoff()      # uncomment to actually run; takes ~30-60s on CPU Ollama
# print(result)

## 5 · AutoGen — same trio as a conversation

In [ ]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

llm_config = {
    "config_list": [{
        "model": "qwen2.5:0.5b-instruct",
        "base_url": "http://localhost:11434/v1",
        "api_key": "ollama",
        "price": [0, 0],     # silence cost warnings
    }],
    "temperature": 0.2,
}

researcher = AssistantAgent(name="Researcher",
    system_message="Find 3 facts about the topic. Reply ONLY with a numbered list.",
    llm_config=llm_config)

writer = AssistantAgent(name="Writer",
    system_message="Take the researcher's bullets and write a 5-sentence summary. No fluff.",
    llm_config=llm_config)

critic = AssistantAgent(name="Critic",
    system_message="Critique the draft in 2 sentences and end with 'TERMINATE'.",
    llm_config=llm_config)

user = UserProxyAgent(name="User", human_input_mode="NEVER",
    is_termination_msg=lambda m: "TERMINATE" in (m.get("content") or ""),
    code_execution_config=False, max_consecutive_auto_reply=1)

chat = GroupChat(agents=[user, researcher, writer, critic], messages=[], max_round=8)
manager = GroupChatManager(groupchat=chat, llm_config=llm_config)

# user.initiate_chat(manager, message="Topic: The Transformer attention mechanism.")  # uncomment to run

**What's happening.** The `GroupChatManager` is itself an LLM that picks **who speaks next** at each round. You give it a roster and a stop condition; it routes. Compared to crewAI's pre-defined task list, AutoGen lets agents **decide** to call each other — more flexible, more failure modes.

## 6 · Tool use — agents calling Python

Both frameworks let you expose Python functions to agents. The agent sees the function's signature in its prompt and decides when to call it.

In [ ]:
# crewAI tool — using the @tool decorator
from crewai_tools import tool

@tool("calculator")
def calculator(expr: str) -> str:
    """Evaluates a Python arithmetic expression like '2*3 + 4'."""
    return str(eval(expr, {"__builtins__": {}}, {}))

mathy = Agent(role="Math Helper", goal="Use the calculator for any arithmetic.",
              backstory="You always show your work.",
              tools=[calculator], llm=local_llm, verbose=True)
print("tool wired:", calculator.name)

In [ ]:
# AutoGen — register a function on the assistant
from autogen import register_function

def calc(expr: str) -> str:
    """Eval an arithmetic expression."""
    return str(eval(expr, {"__builtins__":{}}, {}))

mathy_ag = AssistantAgent(name="Mathy", llm_config=llm_config,
    system_message="Use the calc tool for arithmetic.")

register_function(calc, caller=mathy_ag, executor=user,
                  name="calc", description="Evaluate arithmetic")
print("registered:", "calc")

## 7 · Pattern: sequential pipeline (Researcher → Writer → Critic)

Best for tasks with a **clear ordering**. Already shown above. Cost: 3+ LLM calls per turn. Reliability: high. The output of step `i` is in the **context** of step `i+1`.

## 8 · Pattern: hierarchical (Manager + Workers)

```
              ┌── Manager LLM ──┐
              │                 │
   "fetch X"  ▼                 ▼ "summarise Y"
        Worker 1            Worker 2
```

In **crewAI** flip `process=Process.hierarchical`. The manager LLM decides what tasks to dispatch, in what order, and reads each result before moving on.

In **AutoGen** the `GroupChatManager` plays the same role. With code-execution enabled the manager can also tell a worker to *write and run code*.

**When this wins.** When you don't know up-front what sub-tasks are needed (e.g. "research X" expands into different searches based on what shows up). When you do know, sequential is cheaper and more predictable.

## 9 · Common failure modes

| Symptom | Cause | Fix |
|---|---|---|
| Agents loop forever, repeating themselves | No clear stop condition | `max_round` (AutoGen), `max_iter` (crewAI), `is_termination_msg` |
| Cost explodes | Big context replayed to every agent | Use **summarising memory** between agents; trim per-call context |
| Agents disagree on the goal | System prompts overlap or conflict | One sentence in each: **what the agent must produce, what they must NOT do** |
| Hallucinated tool calls | Tiny model, weak instruction-follow | Use ≥ 7-8B for tool-use, or LangGraph for explicit branching (M33) |
| "Manager" picks the wrong worker every time | Manager prompt is too vague | Give it a strict "if X then call Y" decision rubric |


## 10 · Decision table — crewAI vs AutoGen vs LangGraph

| Need | Pick |
|---|---|
| Quick prototype with named roles | **crewAI** |
| Tool-using agents, code execution sandbox | **AutoGen** |
| Explicit state machine, retries, time-travel | **LangGraph** (M33) |
| Single agent + tools | **LangChain** (M32) — don't multi-agent prematurely |
| Production reliability with audit trails | LangGraph + your own dispatcher |

**The honest take.** Most "agent" use cases in production today are either (a) one well-prompted LLM with a small tool list, or (b) an explicit state machine. Multi-agent shines for **brainstorming, research, debate** — places where back-and-forth genuinely improves the answer.


## ✅ Recap
- Multi-agent = several specialised LLMs that **talk to each other** through tasks or conversation.
- **crewAI** is declarative (roles + tasks). **AutoGen** is programmatic (conversation graph + manager).
- Both frameworks point at any OpenAI-compatible endpoint — Ollama for free, GPT-4o for quality.
- Watch for loops, runaway cost, and goal disagreement.
- For audit-grade workflows, drop down to **LangGraph** (M33).

Next: **M44 — vLLM** (high-throughput LLM inference for production).
